In [ ]:
# %%bash
# pip install -U huggingface_hub

# python -m huggingface_hub.cli download xlangai/AgentNet \
#   --repo-type dataset \
#   --include "win_mac_images/images.z0[1-3]" \
#   --local-dir /content/drive/MyDrive/AgentNet



In [ ]:
import os
from google.colab import drive
import json
import re
from collections import Counter
from tqdm import tqdm
from collections import defaultdict



drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [29]:
repo_id = "xlangai/AgentNet"
base_dir = "/content/drive/MyDrive/AgentNet"
img_dir = f"{base_dir}/win_mac_images"
zip_path = f"{img_dir}/images.zip"
extract_dir = os.path.join(img_dir, "office_images")
task_id_filtered = f"{base_dir}/office_task_ids.json"
task_clean = f"{base_dir}/office_task_ids_clean.json"
traj_file = f"{base_dir}/agentnet_win_mac_18k.jsonl"


os.makedirs(img_dir, exist_ok=True)
print("Target directory:", img_dir)


Target directory: /content/drive/MyDrive/AgentNet/win_mac_images


In [ ]:
# with open(task_id_filtered, "r", encoding="utf-8") as f:
#     task_ids = json.load(f)

# pattern = re.compile(r"^\d{14}_[0-9a-fA-F\-]{36}$")

# clean_ids = []
# bad_ids = []

# for tid in task_ids:
#     if pattern.match(tid):
#         clean_ids.append(tid)
#     else:
#         bad_ids.append(tid)

# print("Valid task_ids:", len(clean_ids))
# print("Invalid entries:", len(bad_ids))

# print("\nBad entries:")
# for x in bad_ids[:10]:
#     print(x)

# clean_path = "/content/drive/MyDrive/AgentNet/office_task_ids_clean.json"

# with open(clean_path, "w", encoding="utf-8") as f:
#     json.dump(clean_ids, f)

# print("Clean task list saved:", clean_path)

Valid task_ids: 2450
Invalid entries: 6

Bad entries:
20241016120602_Open System then Click Date&Time then you can now change it according to your region
20241016121316_How to download and install Google Chrome
20241016121256_How to download and install Brave Browser (Ads Blocker)
20241016120656_Open Windows then click settings after that go to  Background, lock screen, colors after that you can now modify your windows
20241016120540_Open Windows click settings then go to apps then click default after that you can change it
20241015185027_x


In [8]:
with open(task_clean, "r", encoding="utf-8") as f:
    office_task_ids = set(json.load(f))

print("Office tasks:", len(office_task_ids))

Office tasks: 2450


In [21]:
office_images = set()

with open(traj_file, "r", encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line)

        if obj.get("task_id") in office_task_ids:
            for step in obj.get("traj", []):
                img = step.get("image")
                if img:
                    office_images.add(img)

print("Office screenshots:", len(office_images))
print("Sample:", list(office_images)[:5])

Office screenshots: 46118
Sample: ['ca44a537-ccff-4ee0-abc9-276352533229.png', 'ead46d5a-9444-4699-85dc-0b6ebdf52445.png', 'ac09fd7d-e241-40e4-938d-2015df1d3029.png', 'a7c45fcb-7505-4d29-930e-138688cae1f5.png', '6222cdab-b9d1-4653-8488-25ecde32b4bf.png']


In [ ]:
action_counter = Counter()

with open(traj_file, "r", encoding="utf-8") as f:

    for line in f:

        data = json.loads(line)

        for step in data["traj"]:

            code = step["value"]["code"]

            matches = re.findall(r'pyautogui\.([a-zA-Z_]+)', code)

            for action in matches:
                action_counter[action] += 1

            if "terminate" in code:
                action_counter["terminate"] += 1


print("Unique actions:", len(action_counter))
print(sorted(action_counter.keys()))
print(action_counter)

Unique actions: 12
['click', 'doubleClick', 'dragTo', 'hotkey', 'hscroll', 'middleClick', 'moveTo', 'press', 'rightClick', 'scroll', 'terminate', 'write']
Counter({'click': 235648, 'moveTo': 35709, 'write': 31118, 'press': 22308, 'scroll': 19142, 'terminate': 17564, 'dragTo': 16273, 'doubleClick': 8168, 'hotkey': 7182, 'rightClick': 4273, 'hscroll': 546, 'middleClick': 42})


In [ ]:
output_file = f"{base_dir}/agentnet_mind2web_style.json"
# action_map = {
#     "click": "CLICK",
#     "doubleClick": "DOUBLE_CLICK",
#     "rightClick": "RIGHT_CLICK",
#     "middleClick": "MIDDLE_CLICK",
#     "moveTo": "MOVE",
#     "dragTo": "DRAG",
#     "scroll": "SCROLL",
#     "hscroll": "HSCROLL",
#     "write": "TYPE",
#     "press": "PRESS",
#     "hotkey": "HOTKEY",
# }

# Cache to reduce API calls
history_cache = {}

# def extract_history(action_text):

#     if action_text in history_cache:
#         return history_cache[action_text]

#     prompt = f"""
# Convert the GUI action description into a short action history line.

# Rules:
# - Format: ACTION on ELEMENT
# - If typing text: TYPE 'text' in ELEMENT
# - Use uppercase ACTION
# - Do not add explanation

# Examples:

# Action: Click on the "New meeting" button
# Output: CLICK on New meeting button

# Action: Type "hello world" in the chat box
# Output: TYPE 'hello world' in chat box

# Action: Press the Enter key
# Output: PRESS Enter key

# Now convert:

# Action: {action_text}

# Output:
# """

#     response = model.generate_content(
#         prompt,
#         generation_config={"temperature": 0}
#     )
#     result = response.text.strip()

#     history_cache[action_text] = result

#     return result

def extract_history(action_text):
    """
    Convert AgentNet natural action text to a short history line
    without calling any external model.
    """

    text = action_text.strip()

    # remove quotes around UI labels
    text = re.sub(r'"([^"]+)"', r'\1', text)

    # normalize verbs to uppercase
    replacements = {
        "click": "CLICK",
        "double click": "DOUBLE_CLICK",
        "right click": "RIGHT_CLICK",
        "middle click": "MIDDLE_CLICK",
        "type": "TYPE",
        "press": "PRESS",
        "scroll": "SCROLL",
        "drag": "DRAG",
        "move": "MOVE"
    }

    lower = text.lower()

    for k, v in replacements.items():
        if lower.startswith(k):
            return text.replace(text.split()[0], v)

    return text


def parse_code(code):
    """
    Parse pyautogui code into structured GUI actions.
    """

    actions = []

    # -------------------------
    # DRAG (moveTo + dragTo)
    # -------------------------
    move_match = re.search(
        r"moveTo\(.*?x\s*=\s*([\d\.]+).*?y\s*=\s*([\d\.]+).*?\)",
        code
    )

    drag_match = re.search(
        r"dragTo\(.*?x\s*=\s*([\d\.]+).*?y\s*=\s*([\d\.]+).*?\)",
        code
    )

    if move_match and drag_match:

        start_x = float(move_match.group(1))
        start_y = float(move_match.group(2))

        end_x = float(drag_match.group(1))
        end_y = float(drag_match.group(2))

        actions.append({
            "ACTION": "DRAG",
            "MARK": None,
            "VALUE": {
                "start": [start_x, start_y],
                "end": [end_x, end_y]
            }
        })

        return actions


    # -------------------------
    # DOUBLE CLICK
    # -------------------------
    if "doubleClick" in code:
        actions.append({
            "ACTION": "DOUBLE_CLICK",
            "MARK": None,
            "VALUE": None
        })
        return actions


    # -------------------------
    # RIGHT CLICK
    # -------------------------
    if "rightClick" in code:
        actions.append({
            "ACTION": "RIGHT_CLICK",
            "MARK": None,
            "VALUE": None
        })
        return actions


    # -------------------------
    # CLICK
    # -------------------------
    if "click" in code:
        actions.append({
            "ACTION": "CLICK",
            "MARK": None,
            "VALUE": None
        })
        return actions


    # -------------------------
    # TYPE
    # -------------------------
    write_match = re.search(r"write\(['\"](.*?)['\"]\)", code)

    if write_match:
        text = write_match.group(1)

        actions.append({
            "ACTION": "TYPE",
            "MARK": None,
            "VALUE": text
        })

        return actions


    # -------------------------
    # PRESS
    # -------------------------
    press_match = re.search(r"press\(['\"](.*?)['\"]\)", code)

    if press_match:
        key = press_match.group(1)

        actions.append({
            "ACTION": "PRESS",
            "MARK": None,
            "VALUE": key
        })

        return actions


    # -------------------------
    # HOTKEY
    # -------------------------
    if "hotkey" in code:

        keys = re.findall(r"['\"](.*?)['\"]", code)

        actions.append({
            "ACTION": "HOTKEY",
            "MARK": None,
            "VALUE": keys
        })

        return actions


    # -------------------------
    # SCROLL
    # -------------------------
    if "scroll" in code:
        actions.append({
            "ACTION": "SCROLL",
            "MARK": None,
            "VALUE": None
        })

        return actions


    return actions


def build_prompt(instruction, history):

    history_text = "\n".join(f"- {h}" for h in history) if history else "None"

    prompt = f"""<image>
Imagine that you are imitating humans doing GUI navigation step by step.

You can perform actions such as CLICK, DOUBLE_CLICK, RIGHT_CLICK, MIDDLE_CLICK, MOVE, DRAG, SCROLL, HSCROLL, TYPE, PRESS, HOTKEY.

Output format must be:
{{"ACTION": action_type, "MARK": numeric_id, "VALUE": text_or_null}}

Task: {instruction}

Previous actions:
{history_text}

For your convenience, UI elements are labeled with numeric marks.

What is the next action?
"""

    return prompt

In [ ]:
with open(task_clean) as f:
    office_task_ids = set(json.load(f))

print("Filtered office tasks:", len(office_task_ids))

dataset = []
matched_tasks = 0
sample_id = 0
action_stats = defaultdict(int)


with open(traj_file, "r") as f:

    for line in tqdm(f):

        data = json.loads(line)
        task_id = data["task_id"]

        if task_id not in office_task_ids:
            continue

        matched_tasks += 1
        instruction = data["instruction"]
        history = []

        for step in data["traj"]:

            value = step["value"]

            if not value.get("last_step_correct", True):
                continue

            action_text = value["action"]
            code = value["code"]
            image_name = step["image"]

            parsed_actions = parse_code(code)

            for act in parsed_actions:

                action_stats[act["ACTION"]] += 1

                user_prompt = build_prompt(instruction, history)

                sample = {
                    "id": f"AgentNet_SoM_{sample_id}",
                    "image": image_name,
                    "conversations": [
                        {"from": "user", "value": user_prompt},
                        {"from": "assistant", "value": json.dumps(act)}
                    ]
                }

                dataset.append(sample)
                sample_id += 1

                history_line = extract_history(action_text)
                history.append(history_line)

        if data.get("task_completed", False):

            image_name = data["traj"][-1]["image"]

            user_prompt = build_prompt(instruction, history)

            terminate_action = {
                "ACTION": "TERMINATE",
                "MARK": None,
                "VALUE": None
            }

            dataset.append({
                "id": f"AgentNet_SoM_{sample_id}",
                "image": image_name,
                "conversations": [
                    {"from": "user", "value": user_prompt},
                    {"from": "assistant", "value": json.dumps(terminate_action)}
                ]
            })

            sample_id += 1

            # append history AFTER sample creation
            history_line = extract_history(action_text)
            history.append(history_line)

        # terminate action
        if data.get("task_completed", False):

            image_name = data["traj"][-1]["image"]

            user_prompt = build_prompt(instruction, history)

            terminate_action = {
                "ACTION": "TERMINATE",
                "MARK": None,
                "VALUE": None
            }

            dataset.append({
                "id": f"AgentNet_SoM_{sample_id}",
                "image": image_name,
                "conversations": [
                    {"from": "user", "value": user_prompt},
                    {"from": "assistant", "value": json.dumps(terminate_action)}
                ]
            })

            sample_id += 1


        # ------------------------------------------------
        # Add TERMINATE action if task completed
        # ------------------------------------------------

        if data.get("task_completed", False):

            image_name = data["traj"][-1]["image"]

            user_prompt = build_prompt(instruction, history)

            terminate_action = {
                "ACTION": "TERMINATE",
                "MARK": None,
                "VALUE": None
            }

            sample = {
                "id": f"AgentNet_SoM_{sample_id}",
                "image": image_name,
                "conversations": [
                    {
                        "from": "user",
                        "value": user_prompt
                    },
                    {
                        "from": "assistant",
                        "value": json.dumps(terminate_action)
                    }
                ]
            }

            dataset.append(sample)

            sample_id += 1


# ------------------------------------------------
# Save dataset
# ------------------------------------------------

with open(output_file, "w") as f:
    json.dump(dataset, f, indent=2)


print("\nConversion complete.")
print("Matched tasks:", matched_tasks)
print("Total training samples:", len(dataset))
print(action_stats)

Filtered office tasks: 2450


17625it [00:17, 982.10it/s] 



Conversion complete.
Matched tasks: 2463
Total training samples: 40457
defaultdict(<class 'int'>, {'CLICK': 28079, 'PRESS': 1697, 'DRAG': 2071, 'SCROLL': 734, 'DOUBLE_CLICK': 1062, 'RIGHT_CLICK': 815, 'HOTKEY': 581})
